# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the <b>FAIR<sup>2</sup> clinical dataset</b> using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print("\nDataset published:", metadata.datePublished)
print("Version:", metadata.version)
print("Keywords:", ', '.join(str(kw) for kw in metadata.keywords))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset contains several tables (record sets) representing clinical features, hormone levels, imaging, and genetic variants. Croissant schema exposes each entity via an `@id`.

Let's inspect the available record sets and their fields.

In [ ]:
# Retrieve record sets and their @ids
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- {rs['@id']} | {rs['name']} | {rs['description']}")

# For each record set, show available fields and their @id
for rs in record_sets:
    print(f"\nFields in RecordSet @id={rs['@id']} (name={rs['name']}):")
    for field in rs['fields']:
        print(f"  - {field['@id']} | {field['name']} | dataType={field.get('dataType','')} | description={field.get('description','')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

**Croissant schema entities are referenced by their `@id`.**

We'll extract data for each record set found above.

In [ ]:
# List of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns in record set {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"\nNo records found for record set {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering records, normalizing numeric fields, and grouping.

Let's select the first record set (usually Table 1: clinical features), filter by age, normalize values, and group where appropriate. All references use the `@id` for field and record set access.

In [ ]:
# Specify which record set and fields to analyze
clinical_recordset_id = record_set_ids[0]
clinical_df = dataframes[clinical_recordset_id]

# Locate numeric field @id (e.g., age)
fields = [f for f in dataset.record_sets[0]['fields']]
# Find age field by name
age_field = next((f for f in fields if 'age' in f['name'].lower()), None)
numeric_field_id = age_field['@id'] if age_field else clinical_df.columns[0]

# Filtering and normalization
threshold = 18
filtered_df = clinical_df[clinical_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by categorical field: e.g. 'infertility' or 'oligomenorrhea'
# Find group field
group_field = None
for f in fields:
    if f['name'] in ['infertility', 'oligomenorrhea']:
        group_field = f['@id']
        break

if group_field and group_field in clinical_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of ages and relationships with infertility status.

We'll create a histogram of age and a bar plot grouped by infertility.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.histplot(clinical_df[numeric_field_id], bins=5, kde=True)
plt.xlabel('Age at Diagnosis')
plt.title('Distribution of Age at Diagnosis')
plt.show()

# Visualize mean age grouped by infertility status (if available)
if group_field and group_field in clinical_df.columns:
    plt.figure(figsize=(6,4))
    sns.barplot(
        x=group_field,
        y=numeric_field_id,
        data=clinical_df,
        estimator='mean'
    )
    plt.title(f"Mean Age at Diagnosis by {group_field}")
    plt.show()

## 6. Conclusion

- The dataset provides structured clinical, laboratory, imaging, and genetic features for three NC-21OHD female patients.
- Analysis using `mlcroissant` allows convenient selection of record sets and fields using their schema `@id`.
- EDA demonstrates filtering, normalization, and grouping even for small groups, yielding insights such as the range of diagnosed ages and clinical heterogeneity.

Further work may include cross-table joining (using patient IDs) and exploration of hormone/imaging/genotype tables, as well as integration with visualization or statistical modeling tools for qualitative analysis.

<i>Note: For actual schema-driven workflows, always refer to the Croissant schema for exact `@id` values and relationships. Due to small N, caution must be used in statistical generalization.</i>